# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# The 'metadata' property is a Python object. To print the main metadata fields, convert to dict/json.
metadata_dict = dataset.metadata.to_json()
print(f"Dataset name: {metadata_dict['name']}")
print(f"Dataset description: {metadata_dict['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant, data is organized into record sets. Each record set provides one table or view of data, and each field maps to a column. All should be referenced by their `@id` (ID string).

Let's list all record sets in this dataset, along with their field and column IDs.

In [ ]:
# List all available RecordSets, their @id, and their fields/columns
record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets declared at the top-level; inspecting distributions for data tables...")
    # Sometimes, record sets are implicitly determined by data files or distributions

# Commonly, record_sets holds objects with .id, .fields, .columns, etc.
record_set_ids = []
for rec in getattr(dataset.metadata, 'record_sets', []):
    print(f"Record Set @id : {rec.id}")
    record_set_ids.append(rec.id)
    if hasattr(rec, 'fields') and rec.fields:
        print("  Fields:")
        for field in rec.fields:
            print(f"    - {field.id} (label: {getattr(field, 'label', '')})")
    else:
        print("  [No explicit fields listed]")
    if hasattr(rec, 'columns') and rec.columns:
        print("  Columns:")
        for col in rec.columns:
            print(f"    - {col.id} (label: {getattr(col, 'label', '')})")
    print("")

# If there are no explicit record sets, get them via dataset.record_set_ids
if not record_set_ids:
    try:
        record_set_ids = list(dataset.record_set_ids)
        print("Record set IDs found in dataset:", record_set_ids)
    except Exception as e:
        print("Could not extract record sets:", str(e))

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All `record_set` and field parameters must be referenced via their `@id` fields.

We'll demonstrate loading the first available record set, and inspecting its fields and some sample records.

In [ ]:
# Select a record set by its @id (use the first if multiple are present)
if record_set_ids:
    main_record_set_id = record_set_ids[0]
else:
    # Fallback: Try to load without specifying the record set
    main_record_set_id = None

print(f"Attempting to load data from Record Set: {main_record_set_id}")

dataframes = {}
if main_record_set_id:
    # Load records for this record set
    records_iter = dataset.records(record_set=main_record_set_id)
else:
    # Try to load from unnamed main records
    records_iter = dataset.records()

records = list(records_iter)
df = pd.DataFrame(records)
dataframes[main_record_set_id] = df
print(f"Columns for record set {main_record_set_id}:\n", df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. All field names used should reference their `@id`.

We'll select a numeric field for analysis (for instance, 'MW [kDa]' for molecular weight, or 'Coverage [%]' for sequence coverage), filter, normalize, and group by a text category.

**To see available fields and their `@id`, view the columns printed above.**

In [ ]:
# Pick example field IDs from columns (adjust these if your dataset uses different or additional field IDs)
available_columns = df.columns.tolist()
print("Available columns / field @ids:", available_columns)

# Example: Often, columns like 'MW [kDa]' or 'cr:MW_kDa' represent molecular weight.
# For this dataset, try to choose a numeric field from columns:
# Let's try 'cr:MW_kDa' or fallback to 'cr:Coverage_percent'
if 'cr:MW_kDa' in available_columns:
    numeric_field_id = 'cr:MW_kDa'
elif 'cr:Coverage_percent' in available_columns:
    numeric_field_id = 'cr:Coverage_percent'
else:
    numeric_field_id = None

if numeric_field_id:
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    # Remove outliers and filter records (e.g., MW > 10)
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())
    # Normalize the numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())
    # Group by a text/categorical field (e.g., by 'cr:Description' or 'cr:accession' if available)
    if 'cr:Description' in available_columns:
        group_field_id = 'cr:Description'
    elif 'cr:accession' in available_columns:
        group_field_id = 'cr:accession'
    else:
        group_field_id = None
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No suitable numeric field found for EDA. Please check the columns and adjust field @id.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we'll plot a histogram of the chosen numeric field, and a scatter or boxplot if grouping fields are available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    df[numeric_field_id].hist(bins=30, alpha=0.7)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot by group field if available
    if group_field_id and group_field_id in df.columns:
        to_plot = filtered_df[[numeric_field_id, group_field_id]].dropna()
        if not to_plot.empty:
            plt.figure(figsize=(12, 4))
            sns.boxplot(x=group_field_id, y=numeric_field_id, data=to_plot)
            plt.xticks(rotation=90)
            plt.title(f"{numeric_field_id} distribution by {group_field_id}")
            plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded a Croissant-described biological dataset using its URL schema and the `mlcroissant` Python library.
- Inspected the available record sets, fields, and columns using their `@id` references.
- Loaded the main data table into a pandas DataFrame, filtered and normalized a numeric field.
- Grouped data by a categorical field for statistical summary.
- Visualized key field distributions.

This workflow is generalizable to any Croissant-compliant dataset: use entity `@id` references for robust, schema-independent data handling. You are encouraged to repeat these EDA steps with other fields or record sets of interest!